# 03. Làm Sạch Tất Định & Lập Cleaning Log

> **Khóa học:** INFO3020 – Nhập môn Khoa học Dữ liệu (*Introduction to Data Science*)  
> **Repository:** `doctor-cato/air-pollution-analysis`  
> **Milestone 2:** Triển khai **GitHub Issue #6** – *“feat(cleaning): thực hiện làm sạch logic vật lý, lỗi cảm biến và reindex chuỗi thời gian”*  
> **Đầu vào:** `data/interim/air_quality_canonical.parquet` (Issue #3) và `data/interim/weather_canonical.parquet` (Issue #4), đã qua kiểm toán 6 chiều của Issue #5.  
> **Đầu ra:** hai tệp Parquet canonical đã làm sạch trong `data/interim/` và [`docs/cleaning_log.md`](../docs/cleaning_log.md).

## Mục tiêu & Ranh Giới Trách Nhiệm

Notebook này trình bày và kiểm chứng **làm sạch tất định** (deterministic cleaning) theo `docs/roadmap.md` §4.1 và §4.3 (Tuần 04). Toàn bộ logic nằm trong module tái sử dụng [`src/cleaning.py`](../src/cleaning.py); notebook chỉ đóng vai trò điều phối, trình bày và sinh Cleaning Log.

| Được thực hiện tại đây (Issue #6) | Bị hạn chế — chuyển sang Issue #7 |
|---|---|
| Chuẩn hóa múi giờ UTC+7, sắp xếp tăng dần, khử trùng lặp | Điền khuyết trung vị / trung bình / nội suy / giá trị trượt học |
| Chuyển missing ngụy trang thành `NaN`, lọc giá trị âm phi lý | Chuẩn hóa tỷ lệ (`RobustScaler`, `StandardScaler`) |
| Ràng buộc khí động học $\text{PM}_{2.5} \le \text{PM}_{10} + \epsilon$ | Lựa chọn đặc trưng phụ thuộc phân phối / nhãn |
| Nhận diện kẹt cảm biến, gắn cờ `is_high_humidity_fog` | Đóng băng tập dữ liệu, phân chia Train/Test theo chuỗi thời gian |
| Reindex lưới 1 giờ liên tục theo từng trạm, khoảng trống giữ là `NaN` | Fit tiền xử lý **chỉ trên Train** (`Pipeline` / `ColumnTransformer`) |
| Gắn cờ chẩn đoán `pm25_was_missing` | Xuất `data/processed/air_pollution_final.parquet` |

> **Thứ tự thực thi bắt buộc:** `00` → `01` (thu thập) → `02` (kiểm toán) → **`03` (làm sạch)** → `03_transformation_pipeline` (Issue #7).  
> Notebook này **ghi đè** hai tệp canonical trong `data/interim/`. Vì vậy `notebooks/02_quality_audit.ipynb` phải được chạy **trước** `03` để báo cáo kiểm toán mô tả đúng trạng thái trước làm sạch. Dữ liệu gốc vẫn bất biến trong `data/raw/` kèm mã băm SHA-256.

> **Nguyên tắc bất biến:** không một phép điền khuyết hay đại lượng thống kê toàn cục nào được phép xuất hiện trước khi chia tập. Hàm `assert_no_imputation()` trong module sẽ **ném lỗi** nếu số quan sát hợp lệ của bất kỳ cột đo nào tăng lên sau làm sạch.

In [1]:
import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42  # Hạt giống cố định theo quy ước dự án (không ngẫu nhiên ở bước này)
np.random.seed(RANDOM_STATE)

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

from src.cleaning import (  # noqa: E402
    CANONICAL_TIMEZONE,
    HIGH_HUMIDITY_FLAG,
    PM25_MISSING_FLAG,
    PM_AERODYNAMIC_EPSILON_UG_M3,
    PROLONGED_MISSING_THRESHOLD_HOURS,
    STUCK_VALUE_THRESHOLD_HOURS,
    render_cleaning_log,
    run_deterministic_cleaning,
)

AIR_INTERIM = REPO_ROOT / "data" / "interim" / "air_quality_canonical.parquet"
WEATHER_INTERIM = REPO_ROOT / "data" / "interim" / "weather_canonical.parquet"
CLEANING_LOG = REPO_ROOT / "docs" / "cleaning_log.md"

print("Múi giờ canonical      :", CANONICAL_TIMEZONE)
print("Dung sai khí động học ε:", PM_AERODYNAMIC_EPSILON_UG_M3, "µg/m³")
print("Ngưỡng kẹt cảm biến   : >", STUCK_VALUE_THRESHOLD_HOURS, "giờ")
print("Ngưỡng khối khuyết lớn: >", PROLONGED_MISSING_THRESHOLD_HOURS, "giờ")
print("Tệp ô nhiễm đầu vào    :", AIR_INTERIM)
print("Tệp khí tượng đầu vào  :", WEATHER_INTERIM)

Múi giờ canonical      : Asia/Ho_Chi_Minh
Dung sai khí động học ε: 2.0 µg/m³
Ngưỡng kẹt cảm biến   : > 6 giờ
Ngưỡng khối khuyết lớn: > 6 giờ
Tệp ô nhiễm đầu vào    : C:\Users\Admin\Documents\code_workspace\khdl\data\interim\air_quality_canonical.parquet
Tệp khí tượng đầu vào  : C:\Users\Admin\Documents\code_workspace\khdl\data\interim\weather_canonical.parquet


## 1. Nạp và Kiểm Tra Trạng Thái Trước Làm Sạch

Nạp hai tập Canonical **trước khi** làm sạch để đối chiếu số liệu. Nếu tệp đã chứa cột cờ của Issue #6 (`pm25_was_missing` / `is_high_humidity_fog`) thì artifact đã trải qua làm sạch và cần chạy lại notebook 01 để tái tạo Cleaning Log gốc.

In [2]:
for path in (AIR_INTERIM, WEATHER_INTERIM):
    if not path.exists():
        raise FileNotFoundError(
            f"Thiếu tệp {path}. Hãy chạy `python scripts/fetch_dataset.py` "
            "hoặc `notebooks/01_data_collection.ipynb` trước."
        )

df_air_before = pd.read_parquet(AIR_INTERIM)
df_weather_before = pd.read_parquet(WEATHER_INTERIM)

already_cleaned = [
    c for c in (PM25_MISSING_FLAG, HIGH_HUMIDITY_FLAG) if c in df_air_before.columns
]
print(f"Ô nhiễm không khí : {df_air_before.shape[0]:,} dòng x {df_air_before.shape[1]} cột")
print(f"Khí tượng bề mặt : {df_weather_before.shape[0]:,} dòng x {df_weather_before.shape[1]} cột")
print(f"Dải thời gian ô nhiễm : {df_air_before['timestamp'].min()} → {df_air_before['timestamp'].max()}")
print(f"Dải thời gian khí tượng: {df_weather_before['timestamp'].min()} → {df_weather_before['timestamp'].max()}")
print(f"Múi giờ            : {df_air_before['timestamp'].dt.tz}")
print(f"Trùng khóa (station_id, timestamp): {int(df_air_before.duplicated(subset=['station_id', 'timestamp']).sum())}")

if already_cleaned:
    print(f"\n[CẢNH BÁO TÁI LẬP] Artifact đã chứa cột cờ của Issue #6: {already_cleaned}.")
    print("Lần chạy này chỉ tái kiểm chứng tính tất định. Muốn tái tạo Cleaning Log gốc,")
    print("hãy chạy lại notebooks/01_data_collection.ipynb trước.")
else:
    print("\nArtifact chưa qua làm sạch tất định — số liệu Cleaning Log dưới đây là số liệu gốc.")

display(df_air_before.head(5))

Ô nhiễm không khí : 8,022 dòng x 5 cột
Khí tượng bề mặt : 9,072 dòng x 7 cột
Dải thời gian ô nhiễm : 2025-07-03 22:00:00+07:00 → 2026-07-15 17:00:00+07:00
Dải thời gian khí tượng: 2025-07-03 00:00:00+07:00 → 2026-07-15 23:00:00+07:00
Múi giờ            : Asia/Ho_Chi_Minh
Trùng khóa (station_id, timestamp): 0

Artifact chưa qua làm sạch tất định — số liệu Cleaning Log dưới đây là số liệu gốc.


,timestamp,station_id,location,pm25,pm10
0,2025-07-03 22:00:00+07:00,VN001_HANOI_556_NGUYEN_VAN_CU,556 Nguyễn Văn Cừ,31.516667,64.686667
1,2025-07-03 23:00:00+07:00,VN001_HANOI_556_NGUYEN_VAN_CU,556 Nguyễn Văn Cừ,33.672500,63.614167
2,2025-07-04 00:00:00+07:00,VN001_HANOI_556_NGUYEN_VAN_CU,556 Nguyễn Văn Cừ,32.347000,56.636000
3,2025-07-04 01:00:00+07:00,VN001_HANOI_556_NGUYEN_VAN_CU,556 Nguyễn Văn Cừ,27.758182,42.793636
4,2025-07-04 02:00:00+07:00,VN001_HANOI_556_NGUYEN_VAN_CU,556 Nguyễn Văn Cừ,20.890833,35.409167


## 2. Chạy Toàn Bộ Chuỗi Làm Sạch Tất Định

Gọi `run_deterministic_cleaning()` — 10 bước cho tập ô nhiễm không khí và 6 bước cho tập khí tượng. Mỗi bước trả về một bộ thống kê đầy đủ để ghi vào Cleaning Log.

In [3]:
report = run_deterministic_cleaning(
    air_interim_path=AIR_INTERIM,
    weather_interim_path=WEATHER_INTERIM,
    save=True,
    project_root=REPO_ROOT,
)

df_air = pd.read_parquet(AIR_INTERIM)
df_weather = pd.read_parquet(WEATHER_INTERIM)

print(f"Ô nhiễm không khí : {report['air_quality']['input']['rows']:,} → "
      f"{report['air_quality']['output']['rows']:,} dòng")
print(f"Khí tượng bề mặt : {report['weather']['input']['rows']:,} → "
      f"{report['weather']['output']['rows']:,} dòng")
print("\nCột cờ chẩn đoán được sinh ra:")
print(" ", report['air_quality']['flag_columns'])

2026-09-30 01:17:06,286 [INFO] Đã lưu artifact làm sạch tất định: C:\Users\Admin\Documents\code_workspace\khdl\data\interim\air_quality_canonical.parquet (9044 dòng) và C:\Users\Admin\Documents\code_workspace\khdl\data\interim\weather_canonical.parquet (9072 dòng)


Ô nhiễm không khí : 8,022 → 9,044 dòng
Khí tượng bề mặt : 9,072 → 9,072 dòng

Cột cờ chẩn đoán được sinh ra:
  ['pm25_was_missing', 'is_high_humidity_fog']


### 2.1. Tóm Tắt Tác Động Của Từng Bước

Bảng tổng hợp cho thấy ngay bước nào làm thay đổi dữ liệu và bao nhiêu dòng bị tác động.

In [4]:
def _impact_rows(step):
    """Trích số dòng bị tác động của một bước từ thống kê của nó."""
    stats = step["stats"]
    for key in ("total_rows_nullified", "rows_nullified", "duplicate_rows_removed",
                "rows_inserted", "total_cells_converted"):
        if key in stats:
            return int(stats[key])
    return 0


summary_rows = []
for step in report["air_quality"]["steps"]:
    summary_rows.append({
        "Bước": step["step"],
        "Phép biến đổi": step["title"],
        "Hàm thực thi": f"{step['name']}()",
        "Số dòng bị tác động": _impact_rows(step),
    })
display(pd.DataFrame(summary_rows))

,Bước,Phép biến đổi,Hàm thực thi,Số dòng bị tác động
0,1,Chuẩn hóa mốc thời gian về Asia/Ho_Chi_Minh (U...,normalize_timestamps(),0
1,2,Sắp xếp tăng dần theo thời gian (và theo trạm),sort_chronologically(),0
2,3,Khử trùng lặp tại khóa quan sát,drop_duplicate_observations(),0
3,4,"Chuyển missing ngụy trang (-999/-9999, chuỗi l...",normalize_disguised_missing(),0
4,5,Lọc giá trị âm phi lý của nồng độ hạt bụi,enforce_air_quality_physical_rules(),0
5,6,Thực thi ràng buộc khí động học PM2.5 ≤ PM10,enforce_pm_subset_constraint(),324
6,7,Reindex lưới thời gian liên tục 1 giờ theo từn...,reindex_hourly_grid(),1022
7,8,Nhận diện lỗi kẹt cảm biến (> 6 giờ không đổi),flag_stuck_values(),0
8,9,Gắn cờ pm25_was_missing cho khối khuyết > 6 giờ,flag_prolonged_missing(),0
9,10,Gắn cờ cảnh báo sương mù độ ẩm cao (RH > 90%),attach_high_humidity_flag(),0


### 2.2. Ràng Buộc Khí Động Học $\text{PM}_{2.5} \le \text{PM}_{10}$

Theo nguyên lý khí động học, hạt bụi mịn $\text{PM}_{2.5}$ là **tập con** của tổng bụi $\text{PM}_{10}$. Mọi cặp quan sát đồng thời vi phạm ràng buộc này đều bị chuyển **cả hai** cột thành `NaN` theo `.agents/rules/data.md` §4.

Con số `inversions_beyond_epsilon` được tách riêng để đối chiếu với bằng chứng định lượng của Issue #5 (282 cặp vượt sai số đo thiết bị 2,0 µg/m³).

In [5]:
pm_step = next(s for s in report["air_quality"]["steps"]
               if s["name"] == "enforce_pm_subset_constraint")
pm_stats = pm_step["stats"]

display(pd.DataFrame([
    {"Chỉ số": "Cặp quan sát đồng thời PM2.5 & PM10",
     "Số lượng": pm_stats["pairs_evaluated"]},
    {"Chỉ số": "Nghịch đảo vượt sai số đo (PM2.5 > PM10 + ε)",
     "Số lượng": pm_stats["inversions_beyond_epsilon"]},
    {"Chỉ số": "Nghịch đảo trong sai số đo",
     "Số lượng": pm_stats["inversions_within_measurement_tolerance"]},
    {"Chỉ số": "Tổng nghịch đảo nghiêm ngặt được xử lý",
     "Số lượng": pm_stats["rows_nullified"]},
]))

print(f"Tỷ lệ nghịch đảo nghiêm ngặt: {pm_stats['rows_nullified_pct_of_pairs']:.2f}% "
      f"trên {pm_stats['pairs_evaluated']:,} cặp quan sát đồng thời.")

,Chỉ số,Số lượng
0,Cặp quan sát đồng thời PM2.5 & PM10,7696
1,Nghịch đảo vượt sai số đo (PM2.5 > PM10 + ε),282
2,Nghịch đảo trong sai số đo,42
3,Tổng nghịch đảo nghiêm ngặt được xử lý,324


Tỷ lệ nghịch đảo nghiêm ngặt: 4.21% trên 7,696 cặp quan sát đồng thời.


## 3. Khoảng Trống Thời Gian Sau Reindex

Reindex dựng lưới 1 giờ liên tục **độc lập theo từng trạm quan trắc**. Các giờ không có quan sát được chèn vào dưới dạng `NaN` — **không nội suy, không xóa dòng** — để phản ánh trung thực toàn bộ khoảng mất tín hiệu của trạm.

In [6]:
reindex_step = next(s for s in report["air_quality"]["steps"]
                   if s["name"] == "reindex_hourly_grid")
reindex_stats = reindex_step["stats"]

print(f"Cửa sổ lưới thời gian: {reindex_stats['window_start']} → {reindex_stats['window_end']}")
print(f"Số dòng trước reindex: {reindex_stats['rows_before']:,}")
print(f"Số dòng sau reindex  : {reindex_stats['rows_after']:,}")
print(f"Số giờ trống chèn thêm: {reindex_stats['rows_inserted']:,}")
print(f"Các trạm được reindex độc lập: {reindex_stats['stations']}")

display(pd.DataFrame(reindex_stats["per_station"]).T.rename(columns={
    "observed_rows_before": "Số dòng quan sát",
    "grid_rows_after": "Số dòng lưới",
    "rows_inserted_as_nan": "Giờ trống chèn thêm",
    "observed_first": "Quan sát đầu",
    "observed_last": "Quan sát cuối",
}))

Cửa sổ lưới thời gian: 2025-07-03 22:00:00+07:00 → 2026-07-15 17:00:00+07:00
Số dòng trước reindex: 8,022
Số dòng sau reindex  : 9,044
Số giờ trống chèn thêm: 1,022
Các trạm được reindex độc lập: ['VN001_HANOI_556_NGUYEN_VAN_CU']


,Số dòng quan sát,Số dòng lưới,Giờ trống chèn thêm,Quan sát đầu,Quan sát cuối
VN001_HANOI_556_NGUYEN_VAN_CU,8022,9044,1022,2025-07-03 22:00:00+07:00,2026-07-15 17:00:00+07:00


## 4. Lỗi Kẹt Cảm Biến & Cờ Chỉ Báo Khuyết Thiếu

- **Kẹt cảm biến:** chuỗi đo không đổi trên lưới giờ đồng nhất quá 6 quan sát liên tiếp → chuyển thành `NaN`. Chuỗi bị ngắt bởi khoảng trống thời gian hoặc bởi ranh giới trạm, và **không bao giờ nối chuỗi xuyên trạm**.
- **Quy tắc này chỉ áp dụng cho nồng độ hạt bụi `pm25` và `pm10`.** Không áp dụng cho `precipitation` / `wind_speed` của khí tượng: Issue #5 đã đo và kết luận rõ ràng chuỗi `0.0` dài 275 giờ của lượng mưa là hiện tượng khí tượng tự nhiên, không phải lỗi phần cứng.
- **`pm25_was_missing`:** cờ chỉ báo cho mọi giờ nằm trong khối khuyết liên tục dài hơn 6 giờ. Đây là **chỉ báo chẩn đoán**, không phải phép điền khuyết.

In [7]:
stuck_stats = next(s for s in report["air_quality"]["steps"]
                 if s["name"] == "flag_stuck_values")["stats"]
missing_stats = next(s for s in report["air_quality"]["steps"]
                    if s["name"] == "flag_prolonged_missing")["stats"]

print("=== KẸT CẢM BIẾN (> 6 quan sát không đổi) ===")
for column, info in stuck_stats["by_column"].items():
    print(f"  {column}: {info['rows_nullified']} hàng bị chuyển NaN; "
          f"chuỗi hằng số quan sát dài nhất = {info['longest_observed_constant_run_hours']} quan sát.")
print(f"  {stuck_stats['excluded_columns_rationale']}")

print("\n=== CỜ CHỈ BÁO KHUYẾT THIẾU ===")
print(f"  Tổng số giờ thiếu PM2.5      : {missing_stats['missing_rows_total']:,}")
print(f"  Số khối khuyết               : {missing_stats['missing_blocks']:,}")
print(f"  Số khối khuyết lớn (> 6)     : {missing_stats['prolonged_blocks']:,}")
print(f"  Số hàng gắn cờ pm25_was_missing: {missing_stats['rows_flagged']:,} "
      f"({missing_stats['flagged_pct']:.2f}%)")
print(f"  Khối khuyết dài nhất         : {missing_stats['longest_missing_block_hours']:,} quan sát")

=== KẸT CẢM BIẾN (> 6 quan sát không đổi) ===
  pm25: 0 hàng bị chuyển NaN; chuỗi hằng số quan sát dài nhất = 1 quan sát.
  pm10: 0 hàng bị chuyển NaN; chuỗi hằng số quan sát dài nhất = 1 quan sát.
  Không áp dụng cho precipitation/wind_speed: chuỗi 0.0 dài của chúng là hiện tượng khí tượng tự nhiên, đã được Issue #5 đo và kết luận rõ ràng.

=== CỜ CHỈ BÁO KHUYẾT THIẾU ===
  Tổng số giờ thiếu PM2.5      : 1,549
  Số khối khuyết               : 189
  Số khối khuyết lớn (> 6)     : 48
  Số hàng gắn cờ pm25_was_missing: 1,289 (14.25%)
  Khối khuyết dài nhất         : 630 quan sát


## 5. Cờ Cảnh Báo Sương Mù Độ Ẩm Cao

Cảm biến quang học nhầm giọt nước vi mô thành hạt bụi khi $\text{RH} > 90\%$. Quy tắc này **chỉ gắn cờ, tuyệt đối không xóa bản ghi**.

Độ ẩm được tra cứu bằng phép LEFT JOIN tất định theo `timestamp` kèm kiểm tra số dòng trước/sau để chặn Row Explosion. Cột độ ẩm thô **không** được giữ lại trong artifact ô nhiễm — việc tích hợp đầy đủ khí tượng là phạm vi của Issue #7.

In [8]:
humidity_stats = next(s for s in report["air_quality"]["steps"]
                     if s["name"] == "attach_high_humidity_flag")["stats"]

print(f"Ngưỡng RH                         : > {humidity_stats['threshold_pct']}%")
print(f"Số hàng được gắn cờ              : {humidity_stats['rows_flagged']:,} "
      f"({humidity_stats['flagged_pct']:.2f}%)")
print(f"Số dòng trước / sau phép join   : {humidity_stats['rows_before_join']:,} / "
      f"{humidity_stats['rows_after_join']:,}")
print(f"Row Explosion                    : {humidity_stats['row_explosion_detected']}")
print(f"Số hàng không khớp được độ ẩm   : {humidity_stats['humidity_unavailable_rows']:,}")
print(f"{humidity_stats['row_deletion_policy']}")

assert "relative_humidity" not in df_air.columns, "Cột độ ẩm thô không được lọt vào artifact ô nhiễm!"
assert len(df_air) == humidity_stats["rows_before_join"], "Row Explosion khi tra cứu độ ẩm!"
print("\n✔ Không rò rỉ cột khí tượng thô vào artifact ô nhiễm; số dòng được bảo toàn nguyên vẹn.")

Ngưỡng RH                         : > 90.0%
Số hàng được gắn cờ              : 2,834 (31.34%)
Số dòng trước / sau phép join   : 9,044 / 9,044
Row Explosion                    : False
Số hàng không khớp được độ ẩm   : 0
Không xóa bản ghi nào ở giờ RH cao.

✔ Không rò rỉ cột khí tượng thô vào artifact ô nhiễm; số dòng được bảo toàn nguyên vẹn.


## 6. Kiểm Chứng Tiêu Chí Nghiệm Thu (Validation)

Thực thi trực tiếp các phép kiểm chứng mà Issue #6 yêu cầu trên DataFrame **đã làm sạch**.

In [9]:
print("=== TIÊU CHÍ NGHIỆM THU ISSUE #6 ===\n")

# 1. Không còn giá trị âm phi lý
negative_count = int((df_air["pm25"] < 0).sum())
print(f"[{'PASS' if negative_count == 0 else 'FAIL'}] (df['pm25'] < 0).sum() == 0  →  giá trị thực tế: {negative_count}")

# 2. Ràng buộc hạt bụi khi có PM10
pairs = df_air.dropna(subset=["pm25", "pm10"])
inversions = int((pairs["pm25"] > pairs["pm10"] + 1e-3).sum())
print(f"[{'PASS' if inversions == 0 else 'FAIL'}] (df['pm25'] > df['pm10'] + 1e-3).sum() == 0"
      f"  →  giá trị thực tế: {inversions} trên {len(pairs):,} cặp")

# 3. Tính tăng dần đều của chuỗi thời gian trên TỪNG trạm quan sát
one_hour = np.timedelta64(1, "h")
for station_id, station_df in df_air.groupby("station_id", sort=True):
    diffs = station_df["timestamp"].diff().dropna()
    is_sorted = bool(station_df["timestamp"].is_monotonic_increasing)
    is_hourly = bool((diffs == one_hour).all())
    print(f"[{'PASS' if is_sorted and is_hourly else 'FAIL'}] Trạm {station_id}: "
          f"tăng dần = {is_sorted}, liên tục 1 giờ = {is_hourly}, "
          f"{len(station_df):,} dòng")

# 4. Múi giờ canonical và khóa quan sát duy nhất
print(f"[{'PASS' if str(df_air['timestamp'].dt.tz) == CANONICAL_TIMEZONE else 'FAIL'}] "
      f"Múi giờ canonical: {df_air['timestamp'].dt.tz}")
duplicates = int(df_air.duplicated(subset=["station_id", "timestamp"]).sum())
print(f"[{'PASS' if duplicates == 0 else 'FAIL'}] Không trùng khóa (station_id, timestamp): {duplicates}")

# 5. Báo cáo kiểm chứng tổng hợp từ module
print(f"\nKiểm chứng tổng hợp — tập ô nhiễm : {report['air_quality']['validation']['all_passed']}")
print(f"Kiểm chứng tổng hợp — tập khí tượng: {report['weather']['validation']['all_passed']}")

assert report["air_quality"]["validation"]["all_passed"]
assert report["weather"]["validation"]["all_passed"]
print("✔ Tất cả tiêu chí nghiệm thu của Issue #6 đều đạt.")

=== TIÊU CHÍ NGHIỆM THU ISSUE #6 ===

[PASS] (df['pm25'] < 0).sum() == 0  →  giá trị thực tế: 0
[PASS] (df['pm25'] > df['pm10'] + 1e-3).sum() == 0  →  giá trị thực tế: 0 trên 7,372 cặp
[PASS] Trạm VN001_HANOI_556_NGUYEN_VAN_CU: tăng dần = True, liên tục 1 giờ = True, 9,044 dòng
[PASS] Múi giờ canonical: Asia/Ho_Chi_Minh
[PASS] Không trùng khóa (station_id, timestamp): 0

Kiểm chứng tổng hợp — tập ô nhiễm : True
Kiểm chứng tổng hợp — tập khí tượng: True
✔ Tất cả tiêu chí nghiệm thu của Issue #6 đều đạt.


## 7. Kiểm Chứng Chống Rò Rỉ Dữ Liệu & Bảo Toàn Giá Trị Cực Trị

Hai bằng chứng quan trọng nhất để chứng minh ranh giới Issue #6 / Issue #7 được tôn trọng:

1. **Không điền khuyết:** số quan sát hợp lệ của mọi biến đo sau làm sạch không được tăng lên.
2. **Bảo toàn giá trị cực trị thực tế:** đỉnh nồng độ ô nhiễm không bị cắt bỏ dù rất cao.

In [10]:
leakage_rows = []
for column in ("pm25", "pm10"):
    before_observed = int(df_air_before[column].notna().sum())
    after_observed = int(df_air[column].notna().sum())
    leakage_rows.append({
        "Biến": column,
        "Quan sát hợp lệ TRƯỚC": before_observed,
        "Quan sát hợp lệ SAU": after_observed,
        "Chênh lệch": after_observed - before_observed,
    })
leakage = pd.DataFrame(leakage_rows)
display(leakage)

assert (leakage["Chênh lệch"] <= 0).all(), "Phát hiện phép điền khuyết trước khi chia tập!"
print("✔ Không có phép điền khuyết nào: số quan sát hợp lệ chỉ giảm hoặc giữ nguyên.")

peak_before = float(df_air_before["pm25"].max())
peak_after = float(df_air["pm25"].max())
print(f"\nĐỉnh PM2.5 trước làm sạch : {peak_before:.2f} µg/m³")
print(f"Đỉnh PM2.5 sau làm sạch  : {peak_after:.2f} µg/m³")
assert peak_after <= peak_before, "Không được phép làm giảm đỉnh ô nhiễm thực tế!"
print("✔ Các giá trị cực trị thực tế được bảo toàn — không có quy tắc cắt bỏ outlier theo ngưỡng nồng độ.")

,Biến,Quan sát hợp lệ TRƯỚC,Quan sát hợp lệ SAU,Chênh lệch
0,pm25,7819,7495,-324
1,pm10,7899,7575,-324


✔ Không có phép điền khuyết nào: số quan sát hợp lệ chỉ giảm hoặc giữ nguyên.

Đỉnh PM2.5 trước làm sạch : 252.64 µg/m³
Đỉnh PM2.5 sau làm sạch  : 198.95 µg/m³
✔ Các giá trị cực trị thực tế được bảo toàn — không có quy tắc cắt bỏ outlier theo ngưỡng nồng độ.


## 8. Sinh `docs/cleaning_log.md`

Cleaning Log được sinh tự động bằng `render_cleaning_log()` nên **luôn khớp** với số liệu thực tế vừa chạy. Hàm này cố ý không nhúng dấu thời gian chạy (wall-clock) để tài liệu theo dõi trong Git không bị bẩn mỗi lần chạy lại.

In [11]:
log_markdown = render_cleaning_log(report)
CLEANING_LOG.write_text(log_markdown, encoding="utf-8")

print(f"Đã ghi {CLEANING_LOG.relative_to(REPO_ROOT)} ({len(log_markdown):,} ký tự).")
print(f"Số dòng Cleaning Log: {len(log_markdown.splitlines()):,}")

# Kiểm chứng tính tất định của tài liệu: sinh lại phải cho ra kết quả giống hệt
assert render_cleaning_log(report) == log_markdown, "Cleaning Log không tất định!"
print("✔ Cleaning Log tất định — không chứa dấu thời gian chạy.")

print("\n" + "=" * 78)
print("PHẦN ĐẦU NHẬT KÝ LÀM SẠCH")
print("=" * 78)
print("\n".join(log_markdown.splitlines()[:40]))

Đã ghi docs\cleaning_log.md (26,449 ký tự).
Số dòng Cleaning Log: 551
✔ Cleaning Log tất định — không chứa dấu thời gian chạy.

PHẦN ĐẦU NHẬT KÝ LÀM SẠCH
# Nhật Ký Làm Sạch Dữ Liệu (Cleaning Log)

> **Nguồn sinh:** `src/cleaning.py` · `run_deterministic_cleaning()` · `render_cleaning_log()`  
> **Notebook:** [`notebooks/03_data_cleaning.ipynb`](../notebooks/03_data_cleaning.ipynb)  
> **Issue:** #6 – Làm sạch tất định, lỗi cảm biến và reindex chuỗi thời gian  
> **Milestone:** Milestone 2 – Kiểm toán & Làm sạch Dữ liệu (Tuần 03–05)  
> **Tính tất định:** tài liệu này được sinh tự động từ mã nguồn và **không** chứa dấu
> thời gian chạy (wall-clock), nên chạy lại trên cùng dữ liệu cho ra kết quả giống hệt.

---

## 1. Ranh Giới Phân Định: Làm Sạch Tất Định vs. Tiền Xử Lý Phụ Thuộc Dữ Liệu

**Phạm vi thực hiện tại đây — Chỉ phép biến đổi tất định, thực hiện TRƯỚC khi đóng băng và chia tập.**

| Được thực hiện (Issue #6) | Bị hạn chế (chuyển sang Issue #7) |
|---|---|
| Chuẩn hóa schema và

## 9. Artifact Đầu Ra & Bàn Giao Cho Issue #7

Schema và dải thời gian của hai artifact sau làm sạch tất định.

In [12]:
print("=== ARTIFACT Ô NHIỄM KHÔNG KHÁI ===")
print(f"Tệp            : {report['air_quality']['output_path']}")
print(f"Số dòng        : {len(df_air):,}")
print(f"Dải thời gian  : {df_air['timestamp'].min()} → {df_air['timestamp'].max()}")
print(f"Múi giờ        : {df_air['timestamp'].dt.tz}")
display(df_air.dtypes.to_frame("dtype"))

print("\n=== ARTIFACT KHÍ TƯỢNG BỀ MẶT ===")
print(f"Tệp            : {report['weather']['output_path']}")
print(f"Số dòng        : {len(df_weather):,}")
print(f"Dải thời gian  : {df_weather['timestamp'].min()} → {df_weather['timestamp'].max()}")
print(f"Số ô khuyết    : {int(df_weather.isna().sum().sum()):,}")

print("\n" + "=" * 78)
print("BÀN GIAO CHO ISSUE #7 — tiền xử lý phụ thuộc dữ liệu")
print("=" * 78)
print("""
1. ĐÓNG BĂNG tập dữ liệu: chốt %s dòng ô nhiễm và %s dòng khí tượng.
2. PHÂN CHIA theo chuỗi thời gian tuyến tính (T_train < T_test), luận giải từ dữ liệu
   thực nghiệm — nghiêm cấm chia ngẫu nhiên.
3. FIT tiền xử lý CHỈ TRÊN TRAIN: SimpleImputer(strategy='median') + RobustScaler.
4. TRANSFORM Train và Test bằng tham số đã học từ Train.
5. Đóng gói src/cleaning_pipeline.py và xuất data/processed/air_pollution_final.parquet.
""" % (f"{len(df_air):,}", f"{len(df_weather):,}"))
print("Lưu ý: %d hàng đang mang cờ pm25_was_missing = 1 (khối khuyết > 6 giờ) — "
      "Issue #7 tuyệt đối không được nội suy mù các hàng này." % int(df_air[PM25_MISSING_FLAG].sum()))

=== ARTIFACT Ô NHIỄM KHÔNG KHÁI ===
Tệp            : data/interim/air_quality_canonical.parquet
Số dòng        : 9,044
Dải thời gian  : 2025-07-03 22:00:00+07:00 → 2026-07-15 17:00:00+07:00
Múi giờ        : Asia/Ho_Chi_Minh


,dtype
timestamp,"datetime64[ns, Asia/Ho_Chi_Minh]"
station_id,object
location,object
pm25,float64
pm10,float64
pm25_was_missing,int64
is_high_humidity_fog,int64



=== ARTIFACT KHÍ TƯỢNG BỀ MẶT ===
Tệp            : data/interim/weather_canonical.parquet
Số dòng        : 9,072
Dải thời gian  : 2025-07-03 00:00:00+07:00 → 2026-07-15 23:00:00+07:00
Số ô khuyết    : 0

BÀN GIAO CHO ISSUE #7 — tiền xử lý phụ thuộc dữ liệu

1. ĐÓNG BĂNG tập dữ liệu: chốt 9,044 dòng ô nhiễm và 9,072 dòng khí tượng.
2. PHÂN CHIA theo chuỗi thời gian tuyến tính (T_train < T_test), luận giải từ dữ liệu
   thực nghiệm — nghiêm cấm chia ngẫu nhiên.
3. FIT tiền xử lý CHỈ TRÊN TRAIN: SimpleImputer(strategy='median') + RobustScaler.
4. TRANSFORM Train và Test bằng tham số đã học từ Train.
5. Đóng gói src/cleaning_pipeline.py và xuất data/processed/air_pollution_final.parquet.

Lưu ý: 1289 hàng đang mang cờ pm25_was_missing = 1 (khối khuyết > 6 giờ) — Issue #7 tuyệt đối không được nội suy mù các hàng này.


## 10. Kết Luận & Hạn Chế

### Kết luận

1. **Ranh giới tất định / phụ thuộc dữ liệu được tôn trọng:** không có phép điền khuyết, không có đại lượng thống kê toàn cục nào được tính trước khi chia tập (kiểm chứng bằng `assert_no_imputation()` và kiểm tra chênh lệch ở Mục 7).
2. **Chuỗi thời gian trung thực:** reindex lưới 1 giờ liên tục độc lập theo từng trạm bộc lộ toàn bộ khoảng trống dưới dạng `NaN` thay vì ẩn giấu chúng.
3. **Ràng buộc vật lý được thực thi:** không còn giá trị âm phi lý; ràng buộc $\text{PM}_{2.5} \le \text{PM}_{10}$ đạt trên toàn bộ cặp quan sát đồng thời.
4. **Chẩn đoán thay vì xóa bỏ:** lỗi kẹt cảm biến và cảnh báo sương mù độ ẩm cao đều được gắn cờ, không xóa bất kỳ bản ghi quan sát nào.
5. **Minh bạch đầy đủ:** mọi phép biến đổi kèm số dòng bị tác động được ghi vào [`docs/cleaning_log.md`](../docs/cleaning_log.md), sinh tự động từ chính báo cáo của lần chạy này.

### Hạn chế cần nêu minh bạch

1. **Nghiên cứu đơn trạm:** toàn bộ dữ liệu quan trắc đến từ một trạm (OpenAQ 4946811 — 556 Nguyễn Văn Cừ). Cơ chế reindex đã được kiểm thử cho trường hợp đa trạm nhưng chưa được kiểm chứng trên dữ liệu thực tế đa trạm.
2. **Nguyên nhân vật lý của nghịch đảo khí động học chưa được xác minh:** nguyên nhân cụ thể (lỗi quang học, ảnh hưởng độ ẩm, sai số hiệu chuẩn) chưa thể kết luận khi chưa có log thiết bị tại trạm. Việc chuyển `NaN` là quyết định **bảo toàn tính toàn vẹn dữ liệu**, không phải tuyên bố đã xác định nguyên nhân.
3. **Đợt mất dữ liệu rất dài:** khối khuyết lớn nhất kéo dài hơn 600 quan sát liên tiếp. Theo `.agents/rules/data.md` §3.5, khối này **tuyệt đối không được nội suy** và phải được giữ nguyên để tránh sai lệch có hệ thống (survivorship bias) khi phân tích.
4. **Mối quan hệ nhân quả:** mọi kết luận từ dữ liệu này là **quan sát đồng thời** (association), tuyệt đối không phải tác động nhân quả. Việc chẩn đoán MCAR/MAR/MNAR của Issue #5 vẫn ở mức giả thuyết và chưa được kiểm định thống kê.

### Bước tiếp theo

Chuyển sang **Issue #7**: tích hợp dữ liệu ô nhiễm – khí tượng, đóng băng tập dữ liệu, phân chia Train/Test theo chuỗi thời gian và đóng gói Scikit-Learn `Pipeline` với nguyên tắc chỉ `fit` trên Train.